## Loading Dataset

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split


# 0. Download dataset (run once, skip if already downloaded)

# !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
# !unzip -q chest-xray-pneumonia.zip -d chest_xray_data

DATA_DIR = "chest_xray_data/chest_xray"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
SUBSET_PER_CLASS = 800   # target images per class for the baseline CNN subset

### Preprocessing

In [ ]:
# ------------------------------------------------------------------
# 1. Check class balance across original folders
# ------------------------------------------------------------------
print("Class counts per split:")
for split in ["train", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        path = f"{DATA_DIR}/{split}/{cls}"
        print(f"  {split}/{cls}: {len(os.listdir(path))}")


# 2. Build a stratified subset from train

def gather_filepaths(split):
    filepaths, labels = [], []
    for label, cls in enumerate(["NORMAL", "PNEUMONIA"]):  # 0=NORMAL, 1=PNEUMONIA
        cls_dir = f"{DATA_DIR}/{split}/{cls}"
        for fname in os.listdir(cls_dir):
            filepaths.append(os.path.join(cls_dir, fname))
            labels.append(label)
    return np.array(filepaths), np.array(labels)

train_files_all, train_labels_all = gather_filepaths("train")

# Stratified subset: take up to SUBSET_PER_CLASS images per class
subset_files, subset_labels = [], []
rng = np.random.default_rng(SEED)
for label in [0, 1]:
    idx = np.where(train_labels_all == label)[0]
    n = min(SUBSET_PER_CLASS, len(idx))
    chosen = rng.choice(idx, size=n, replace=False)
    subset_files.extend(train_files_all[chosen])
    subset_labels.extend(train_labels_all[chosen])

subset_files = np.array(subset_files)
subset_labels = np.array(subset_labels)
print(f"\nSubset size: {len(subset_files)} images "
      f"({np.sum(subset_labels==0)} NORMAL, {np.sum(subset_labels==1)} PNEUMONIA)")


# 3. Stratified 80/20 train/val split of the subset

train_files, val_files, train_labels, val_labels = train_test_split(
    subset_files, subset_labels,
    test_size=0.2, stratify=subset_labels, random_state=SEED
)
print(f"Train: {len(train_files)}  Val: {len(val_files)}")


# 4. Build tf.data pipelines from file paths

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0  # normalize to [0,1]
    return img, tf.expand_dims(tf.cast(label, tf.float32), axis=-1)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def make_dataset(filepaths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=SEED)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_files, train_labels, training=True)
val_ds = make_dataset(val_files, val_labels, training=False)


# 5. Test set: use the FULL official test/ folder

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)
class_names = test_ds_raw.class_names
test_ds = test_ds_raw.map(lambda x, y: (x / 255.0, y)).prefetch(tf.data.AUTOTUNE)

print("\nClass names (test set):", class_names)
print("Pipeline ready: train_ds, val_ds, test_ds")

Class counts per split:
  train/NORMAL: 1341
  train/PNEUMONIA: 3875
  test/NORMAL: 234
  test/PNEUMONIA: 390

Subset size: 1600 images (800 NORMAL, 800 PNEUMONIA)
Train: 1280  Val: 320
Found 624 files belonging to 2 classes.

Class names (test set): ['NORMAL', 'PNEUMONIA']
Pipeline ready: train_ds, val_ds, test_ds


### Baseline CNN

In [ ]:
from tensorflow.keras import models, layers, optimizers
import numpy as np

def build_baseline_cnn(input_shape=(224, 224, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')  # binary output: NORMAL vs PNEUMONIA
    ])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_model(model, train_ds, val_ds, epochs=10, run_name="baseline_run1"):
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[early_stop],
        verbose=1
    )
    print(f"\n{run_name} finished after {len(history.history['loss'])} epochs")
    return history


# Run 1

tf.random.set_seed(42)
np.random.seed(42)

baseline_model = build_baseline_cnn()
baseline_model.summary()

history_baseline_run1 = train_model(baseline_model, train_ds, val_ds,
                                     epochs=10, run_name="Baseline CNN - Run 1")

W0000 00:00:1784771921.960360    8586 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1784771922.019534    8586 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1784771922.044313    8586 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
W0000 00:00:1784771923.286028    8586 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1784771924.469157   24327 cpu_allocator_impl.cc:82] Allocation of 14418432 exceeds 10% of free system memory.
I0000 00:00:1784771934.224359   24347 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 883 of 1280
I0000 00:00:1784771937.809489   24347 shuffle_dataset_op.cc:483] Shuffle buffer filled.


40/40 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.6453 - loss: 0.6346 - val_accuracy: 0.7531 - val_loss: 0.5310
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.8219 - loss: 0.4607 - val_accuracy: 0.8656 - val_loss: 0.3826
Epoch 3/10
29/40 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.8502 - loss: 0.3447

KeyboardInterrupt: 